# Train BPE Tokenizers & Tokenize Data

Trains BPE tokenizers with different vocab sizes (8K, 32K, 128K) and tokenizes the dataset.

**Features:**
- Byte-level BPE (byte fallback, no UNK tokens)
- Special tokens: `<|endoftext|>`, `<|padding|>`
- Saves tokenizers in HuggingFace format
- Tokenizes train/val/test splits

In [1]:
from pathlib import Path
from datasets import load_from_disk
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors
from tokenizers.normalizers import NFC
from transformers import PreTrainedTokenizerFast
from tqdm.auto import tqdm

/Users/ahmetcanyavuz/Developer/lm-trainer/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
# ===================
# CONFIGURATION
# ===================

# Raw data path (from notebook 01)
RAW_DATA_DIR = Path("../data/fineweb-edu-raw")

# Output directories
TOKENIZER_DIR = Path("../tokenizers")
DATA_DIR = Path("../data")

# Vocab sizes to train
VOCAB_SIZES = [8_000, 32_000, 128_000]

# Special tokens
SPECIAL_TOKENS = [
    "<|endoftext|>",  # EOS token (id=0)
    "<|padding|>",    # PAD token (id=1)
]

# Number of processes
NUM_PROC = 8

## 1. Load Raw Data

In [3]:
train_raw = load_from_disk(str(RAW_DATA_DIR / "train"))
val_raw = load_from_disk(str(RAW_DATA_DIR / "val"))
test_raw = load_from_disk(str(RAW_DATA_DIR / "test"))

print(f"Train: {len(train_raw):,} documents")
print(f"Val:   {len(val_raw):,} documents")
print(f"Test:  {len(test_raw):,} documents")

Train: 1,800,578 documents
Val:   47,384 documents
Test:  47,384 documents


In [4]:
# Create text iterator for tokenizer training (uses train set only)
def get_training_corpus(batch_size=1000):
    """Yields batches of text for tokenizer training."""
    for i in range(0, len(train_raw), batch_size):
        yield train_raw[i:i+batch_size]["text"]

## 2. Train BPE Tokenizers

Using byte-level BPE (like GPT-2/Llama):
- Byte fallback: any byte sequence can be encoded
- No UNK tokens ever
- Handles any Unicode text

In [5]:
def train_bpe_tokenizer(vocab_size: int, corpus_iterator) -> Tokenizer:
    """Train a byte-level BPE tokenizer."""
    
    # Initialize byte-level BPE model
    tokenizer = Tokenizer(models.BPE())
    
    # Normalizer: NFC unicode normalization
    tokenizer.normalizer = NFC()
    
    # Pre-tokenizer: Byte-level (like GPT-2)
    # This maps bytes to unicode chars, enabling byte fallback
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    
    # Decoder: Byte-level decoder
    tokenizer.decoder = decoders.ByteLevel()
    
    # Trainer
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=SPECIAL_TOKENS,
        min_frequency=2,
        show_progress=True,
    )
    
    # Train
    tokenizer.train_from_iterator(corpus_iterator, trainer=trainer)
    
    # Post-processor: add EOS handling
    tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)
    
    return tokenizer

In [6]:
def save_tokenizer(tokenizer: Tokenizer, vocab_size: int, output_dir: Path):
    """Save tokenizer in HuggingFace format."""
    
    name = f"bpe-{vocab_size // 1000}k"
    save_path = output_dir / name
    save_path.mkdir(parents=True, exist_ok=True)
    
    # Wrap in PreTrainedTokenizerFast for HF compatibility
    hf_tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=tokenizer,
        eos_token="<|endoftext|>",
        pad_token="<|padding|>",
        bos_token="<|endoftext|>",  # Use same as EOS
    )
    
    hf_tokenizer.save_pretrained(save_path)
    print(f"Saved to {save_path}")
    
    return save_path, hf_tokenizer

In [7]:
# Train tokenizers for each vocab size
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
trained_tokenizers = {}

for vocab_size in VOCAB_SIZES:
    print(f"\n{'='*50}")
    print(f"Training BPE tokenizer with vocab_size={vocab_size:,}")
    print(f"{'='*50}")
    
    # Train
    tokenizer = train_bpe_tokenizer(vocab_size, get_training_corpus())
    
    # Save
    save_path, hf_tokenizer = save_tokenizer(tokenizer, vocab_size, TOKENIZER_DIR)
    trained_tokenizers[vocab_size] = (save_path, hf_tokenizer)
    
    # Quick test
    test_text = "Hello, world! This is a test. 你好世界 🎉"
    tokens = hf_tokenizer.encode(test_text)
    decoded = hf_tokenizer.decode(tokens)
    print(f"Test: '{test_text}'")
    print(f"Tokens ({len(tokens)}): {tokens[:20]}...")
    print(f"Decoded: '{decoded}'")
    print(f"Vocab size: {len(hf_tokenizer):,}")


Training BPE tokenizer with vocab_size=8,000



Saved to ../tokenizers/bpe-8k
Test: 'Hello, world! This is a test. 你好世界 🎉'
Tokens (29): [41, 428, 80, 13, 874, 2, 654, 263, 210, 1027, 15, 174, 159, 122, 207, 160, 99, 122, 159, 117]...
Decoded: 'Hello, world! This is a test. 你好世界 🎉'
Vocab size: 8,000

Training BPE tokenizer with vocab_size=32,000



Saved to ../tokenizers/bpe-32k
Test: 'Hello, world! This is a test. 你好世界 🎉'
Tokens (27): [41, 13675, 13, 874, 2, 654, 263, 210, 1027, 15, 174, 159, 122, 207, 160, 99, 122, 25816, 197, 162]...
Decoded: 'Hello, world! This is a test. 你好世界 🎉'
Vocab size: 32,000

Training BPE tokenizer with vocab_size=128,000



Saved to ../tokenizers/bpe-128k
Test: 'Hello, world! This is a test. 你好世界 🎉'
Tokens (23): [35051, 13, 874, 2, 654, 263, 210, 1027, 15, 174, 60650, 207, 102709, 122, 25816, 197, 127355, 187, 174, 170]...
Decoded: 'Hello, world! This is a test. 你好世界 🎉'
Vocab size: 128,000


## 3. Tokenize Train/Val/Test Sets

In [8]:
def tokenize_dataset(dataset, tokenizer, num_proc=8):
    """Tokenize a dataset, keeping uid."""
    
    def tokenize_fn(examples):
        tokens = tokenizer(
            examples["text"],
            add_special_tokens=False,
            truncation=False,
            return_attention_mask=False,
        )
        return {
            "input_ids": tokens["input_ids"],
            "uid": examples["uid"],
        }
    
    return dataset.map(
        tokenize_fn,
        batched=True,
        num_proc=num_proc,
        remove_columns=["text"],
        desc="Tokenizing",
    )

In [9]:
# Tokenize with each tokenizer
for vocab_size, (tok_path, tokenizer) in trained_tokenizers.items():
    print(f"\n{'='*50}")
    print(f"Tokenizing with bpe-{vocab_size // 1000}k")
    print(f"{'='*50}")
    
    output_dir = DATA_DIR / f"fineweb-edu-bpe-{vocab_size // 1000}k"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Tokenize each split
    print("Tokenizing train...")
    train_tok = tokenize_dataset(train_raw, tokenizer, NUM_PROC)
    train_tok.save_to_disk(output_dir / "train")
    
    print("Tokenizing val...")
    val_tok = tokenize_dataset(val_raw, tokenizer, NUM_PROC)
    val_tok.save_to_disk(output_dir / "val")
    
    print("Tokenizing test...")
    test_tok = tokenize_dataset(test_raw, tokenizer, NUM_PROC)
    test_tok.save_to_disk(output_dir / "test")
    
    # Stats
    train_tokens = sum(len(x) for x in train_tok["input_ids"])
    val_tokens = sum(len(x) for x in val_tok["input_ids"])
    test_tokens = sum(len(x) for x in test_tok["input_ids"])
    
    print(f"\nSaved to {output_dir}")
    print(f"  Train: {train_tokens:,} tokens ({train_tokens/1e9:.2f}B)")
    print(f"  Val:   {val_tokens:,} tokens ({val_tokens/1e6:.0f}M)")
    print(f"  Test:  {test_tokens:,} tokens ({test_tokens/1e6:.0f}M)")


Tokenizing with bpe-8k
Tokenizing train...


Saving the dataset (19/19 shards): 100%|██████████| 1800578/1800578 [00:01<00:00, 954215.08 examples/s] 


Tokenizing val...


Saving the dataset (1/1 shards): 100%|██████████| 47384/47384 [00:00<00:00, 744539.48 examples/s]


Tokenizing test...


...

Tokenizing val...


Saving the dataset (1/1 shards): 100%|██████████| 47384/47384 [00:00<00:00, 324439.01 examples/s]


Tokenizing test...


Saving the dataset (1/1 shards): 100%|██████████| 47384/47384 [00:00<00:00, 426605.92 examples/s]



Saved to ../data/fineweb-edu-bpe-128k
  Train: 1,785,654,947 tokens (1.79B)
  Val:   47,034,464 tokens (47M)
  Test:  46,859,565 tokens (47M)


## 4. Summary

In [10]:
print("\n" + "="*60)
print("TOKENIZER TRAINING COMPLETE")
print("="*60)
print(f"\nTokenizers saved to: {TOKENIZER_DIR.absolute()}")
print(f"Tokenized data saved to: {DATA_DIR.absolute()}")
print("\nCreated:")

for vocab_size in VOCAB_SIZES:
    name = f"bpe-{vocab_size // 1000}k"
    print(f"\n  {name}:")
    print(f"    Tokenizer: tokenizers/{name}/")
    print(f"    Data:      data/fineweb-edu-{name}/")


TOKENIZER TRAINING COMPLETE

Tokenizers saved to: /Users/ahmetcanyavuz/Developer/lm-trainer/notebooks/../tokenizers
Tokenized data saved to: /Users/ahmetcanyavuz/Developer/lm-trainer/notebooks/../data

Created:

  bpe-8k:
    Tokenizer: tokenizers/bpe-8k/
    Data:      data/fineweb-edu-bpe-8k/

  bpe-32k:
    Tokenizer: tokenizers/bpe-32k/
    Data:      data/fineweb-edu-bpe-32k/

  bpe-128k:
    Tokenizer: tokenizers/bpe-128k/
    Data:      data/fineweb-edu-bpe-128k/


## 5. Verify Tokenizers

In [11]:
from transformers import AutoTokenizer

# Test loading and using each tokenizer
test_texts = [
    "Hello, world!",
    "The quick brown fox jumps over the lazy dog.",
    "def fibonacci(n):\n    return n if n < 2 else fibonacci(n-1) + fibonacci(n-2)",
    "日本語テスト",
    "🚀🎉👍",
]

for vocab_size in VOCAB_SIZES:
    name = f"bpe-{vocab_size // 1000}k"
    tok = AutoTokenizer.from_pretrained(TOKENIZER_DIR / name)
    
    print(f"\n{name} (vocab={len(tok):,})")
    print("-" * 40)
    
    for text in test_texts:
        tokens = tok.encode(text, add_special_tokens=False)
        print(f"{len(tokens):3d} tokens: {text[:50]}")


bpe-8k (vocab=8,000)
----------------------------------------
  6 tokens: Hello, world!
 14 tokens: The quick brown fox jumps over the lazy dog.
 38 tokens: def fibonacci(n):
    return n if n < 2 else fibon
 18 tokens: 日本語テスト
 12 tokens: 🚀🎉👍

bpe-32k (vocab=32,000)
----------------------------------------
  5 tokens: Hello, world!
 10 tokens: The quick brown fox jumps over the lazy dog.
 38 tokens: def fibonacci(n):
    return n if n < 2 else fibon
 15 tokens: 日本語テスト
 12 tokens: 🚀🎉👍

bpe-128k (vocab=128,000)
----------------------------------------
  4 tokens: Hello, world!
 10 tokens: The quick brown fox jumps over the lazy dog.
 31 tokens: def fibonacci(n):
    return n if n < 2 else fibon
  9 tokens: 日本語テスト
 12 tokens: 🚀🎉👍


## Next Steps

To train a model with a specific tokenizer:

```yaml
# config.yaml
paths:
  tokenizer: ./tokenizers/bpe-32k
  train_data: ./data/fineweb-edu-bpe-32k/train
  val_data: ./data/fineweb-edu-bpe-32k/val
  test_data: ./data/fineweb-edu-bpe-32k/test
```

```bash
python -m src.train config.yaml
```